# Higher-Quality Text-to-Video with ComfyUI + Wan 2.2 5B (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harekrishnashah13/website_deployemnt/blob/claude/video-generation-text-039bcd/notebooks/ComfyUI_Wan2.2_5B_Text_to_Video_Colab.ipynb)

Runs the **"Wan 2.2 5B Video Generation" (TI2V)** workflow from the open-source
[`Comfy-Org/workflow_templates`](https://github.com/Comfy-Org/workflow_templates)
repository on top of [ComfyUI](https://github.com/comfyanonymous/ComfyUI), used here in pure
text-to-video mode (no reference image).

This replaces an earlier version of this notebook that used the smaller Wan 2.1 1.3B model —
that model is very blurry/low-detail by design (it's the *lightest* model that fits a free
T4 GPU). Wan 2.2's 5B model (~18GB of weights) produces noticeably sharper, more coherent video,
**but it does not reliably fit on Colab's free T4 GPU (15GB VRAM).**

**GPU requirement:** `Runtime` -> `Change runtime type` -> **A100** GPU (or **L4** as a cheaper
fallback). Both require **Colab Pro / Pay-As-You-Go compute units** — this step is no longer free.
If you want to stay on the free tier, use the earlier 1.3B version of this notebook instead and
accept its lower quality ceiling.

Pre-loaded with the same two-scene cinematic desert-door script (9:16), generates each scene as
its own clip, and stitches them with a crossfade transition.

**Honest limits, still:** even at 5B this is a small-to-mid open model, not a paid API model like
Kling/Seedance/Veo — expect a solid step up in sharpness and motion coherence, not photoreal
studio quality. The final "4K" step is still a pixel upscale for delivery, not generated detail,
but with real source detail from the bigger model it will look meaningfully better than before.


## 1. Check GPU

In [ ]:
import subprocess

out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                      capture_output=True, text=True).stdout.strip()
print(out)

name = out.split(',')[0].strip() if out else ''
if 'T4' in name:
    print("\nWARNING: You're on a T4 (15GB VRAM). The Wan 2.2 5B model may fail to load or "
          "run out of memory. Go to Runtime -> Change runtime type and select A100 or L4 "
          "(requires Colab Pro / Pay-As-You-Go).")


## 2. Install ComfyUI

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!apt-get -qq install -y ffmpeg > /dev/null


## 3. Download the Wan 2.2 5B (TI2V) models

Model files and URLs are taken directly from the
[`video_wan2_2_5B_ti2v.json`](https://github.com/Comfy-Org/workflow_templates/blob/main/templates/video_wan2_2_5B_ti2v.json)
workflow template (~18GB total).

In [ ]:
import os

os.makedirs('models/diffusion_models', exist_ok=True)
os.makedirs('models/text_encoders', exist_ok=True)
os.makedirs('models/vae', exist_ok=True)

MODELS = {
    'models/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors',
    'models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors',
    'models/vae/wan2.2_vae.safetensors':
        'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors',
}

for dest, url in MODELS.items():
    if os.path.exists(dest) and os.path.getsize(dest) > 1_000_000:
        print('Already downloaded:', dest)
        continue
    print('Downloading', dest, '...')
    !wget -q --show-progress -O "{dest}" "{url}"


## 4. Your script

Pre-filled with the two-scene desert-door concept. Edit freely for a different video.

In [ ]:
scene1_prompt = "Cinematic realism, high-contrast dramatic lighting, vertical 9:16 composition. Smooth transition from a dark star-filled night sky down to a vast rippled desert dune at dawn. Low-angle tracking shot following a woman's silhouette as she walks up the dune, her figure sharp against the bright horizon. Wind whips sand around her feet, her steps deliberate and heavy. Dawn light breaks, casting long shadows and illuminating textured sand, atmospheric wind particles."  #@param {type:"string"}

scene2_prompt = "Cinematic realism, high-contrast dramatic lighting, vertical 9:16 composition. A woman crests a desert sand dune at dawn and stops abruptly, her silhouette sharp against the bright horizon. Camera smoothly pans to reveal a colossal ancient stone door half-buried in sand, its weathered surface etched with mysterious carvings. Dramatic scale contrast between the dark stone and the bright sandy environment, atmospheric wind particles."  #@param {type:"string"}

negative_prompt = "landscape orientation, low contrast, flat lighting, blurry, low quality, static, worst quality, deformed, disfigured, extra fingers, poorly drawn hands, poorly drawn face, cluttered background"  #@param {type:"string"}

# 9:16 vertical, close to Wan 2.2 5B's native 1280x704 bucket
width = 720  #@param {type:"integer"}
height = 1280  #@param {type:"integer"}
frames_per_scene = 81  #@param {type:"integer"}
fps = 24  #@param {type:"integer"}
steps = 30  #@param {type:"integer"}
cfg = 5  #@param {type:"number"}
seed = -1  #@param {type:"integer"}

# Upscale the final stitched video to a 4K-sized (2160x3840) canvas. This is a plain pixel
# upscale for delivery purposes, not real 4K generation detail.
upscale_to_4k = True  #@param {type:"boolean"}


## 5. Start the ComfyUI server

In [ ]:
import subprocess, time, requests

server_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '127.0.0.1', '--port', '8188'],
    cwd='/content/ComfyUI', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for _ in range(120):
    try:
        r = requests.get('http://127.0.0.1:8188/system_stats', timeout=2)
        if r.status_code == 200:
            print('ComfyUI server is up.')
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(2)
else:
    out = server_proc.stdout.read() if server_proc.stdout else ''
    raise RuntimeError('ComfyUI server did not start in time.\n' + out)


## 6. Generate each scene

Builds the `video_wan2_2_5B_ti2v` API graph (text-only, no start image) and queues it once per scene.

In [ ]:
import os, random, uuid, time, requests

def generate_scene(prompt, negative, run_seed):
    workflow = {
        "37": {"class_type": "UNETLoader", "inputs": {
            "unet_name": "wan2.2_ti2v_5B_fp16.safetensors", "weight_dtype": "default"}},
        "38": {"class_type": "CLIPLoader", "inputs": {
            "clip_name": "umt5_xxl_fp8_e4m3fn_scaled.safetensors", "type": "wan", "device": "default"}},
        "39": {"class_type": "VAELoader", "inputs": {"vae_name": "wan2.2_vae.safetensors"}},
        "55": {"class_type": "Wan22ImageToVideoLatent", "inputs": {
            "width": width, "height": height, "length": frames_per_scene, "batch_size": 1,
            "vae": ["39", 0]}},
        "6": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["38", 0]}},
        "7": {"class_type": "CLIPTextEncode", "inputs": {"text": negative, "clip": ["38", 0]}},
        "48": {"class_type": "ModelSamplingSD3", "inputs": {"shift": 8, "model": ["37", 0]}},
        "3": {"class_type": "KSampler", "inputs": {
            "seed": run_seed, "steps": steps, "cfg": cfg, "sampler_name": "uni_pc",
            "scheduler": "simple", "denoise": 1,
            "model": ["48", 0], "positive": ["6", 0], "negative": ["7", 0], "latent_image": ["55", 0]}},
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["3", 0], "vae": ["39", 0]}},
        "57": {"class_type": "CreateVideo", "inputs": {"images": ["8", 0], "fps": fps}},
        "58": {"class_type": "SaveVideo", "inputs": {
            "video": ["57", 0], "filename_prefix": "video/ComfyUI", "format": "auto", "codec": "auto"}},
    }

    client_id = str(uuid.uuid4())
    resp = requests.post('http://127.0.0.1:8188/prompt', json={'prompt': workflow, 'client_id': client_id})
    resp.raise_for_status()
    prompt_id = resp.json()['prompt_id']
    print('Queued prompt:', prompt_id)

    waited, max_wait_s = 0, 2400
    while waited < max_wait_s:
        time.sleep(5)
        waited += 5
        hist = requests.get(f'http://127.0.0.1:8188/history/{prompt_id}').json()
        entry = hist.get(prompt_id)
        if not entry:
            continue
        status = entry.get('status', {})
        if status.get('status_str') == 'error':
            raise RuntimeError(f'Generation failed: {status}')
        if status.get('completed'):
            for node_out in entry.get('outputs', {}).values():
                for val in node_out.values():
                    if isinstance(val, list):
                        for item in val:
                            if isinstance(item, dict) and 'filename' in item:
                                subfolder = item.get('subfolder', '')
                                return os.path.join('/content/ComfyUI/output', subfolder, item['filename'])
            raise RuntimeError('Completed but no output file found.')
    raise RuntimeError('Timed out waiting for video generation.')


base_seed = seed if seed is not None and seed >= 0 else random.randint(0, 2**32 - 1)

print('Generating Scene 1/2 ...')
scene1_path = generate_scene(scene1_prompt, negative_prompt, base_seed)
print('Scene 1 done:', scene1_path)

print('Generating Scene 2/2 ...')
scene2_path = generate_scene(scene2_prompt, negative_prompt, base_seed + 1)
print('Scene 2 done:', scene2_path)


## 7. Stitch scenes with a crossfade transition

Uses `ffmpeg`'s `xfade` filter for a smooth cinematic cut between Scene 1 and Scene 2.

In [ ]:
import subprocess, json as _json

def probe_duration(path):
    out = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'json', path], capture_output=True, text=True
    )
    return float(_json.loads(out.stdout)['format']['duration'])

transition_duration = min(1.0, probe_duration(scene1_path) * 0.25)  #@param {type:"number"}
scene1_dur = probe_duration(scene1_path)
offset = max(0.0, scene1_dur - transition_duration)

stitched_path = '/content/ComfyUI/output/final_stitched.mp4'
filter_complex = (
    f"[0:v][1:v]xfade=transition=fade:duration={transition_duration}:offset={offset},"
    f"format=yuv420p[v]"
)

cmd = [
    'ffmpeg', '-y',
    '-i', scene1_path, '-i', scene2_path,
    '-filter_complex', filter_complex,
    '-map', '[v]',
    stitched_path,
]
subprocess.run(cmd, check=True)
print('Stitched video:', stitched_path)


## 8. (Optional) Upscale to a 4K-sized canvas

Plain Lanczos pixel upscale to 2160x3840 for delivery. Since the 5B model has much more real
detail than the earlier 1.3B version, this will hold up noticeably better than before — but it's
still upscaling, not genuine 4K generation.

In [ ]:
final_path = stitched_path

if upscale_to_4k:
    upscaled_path = '/content/ComfyUI/output/final_stitched_4k.mp4'
    subprocess.run([
        'ffmpeg', '-y', '-i', stitched_path,
        '-vf', 'scale=2160:3840:flags=lanczos',
        upscaled_path,
    ], check=True)
    final_path = upscaled_path

print('Final video:', final_path)


## 9. Preview and download

In [ ]:
from IPython.display import Video, display

display(Video(final_path, embed=True))

try:
    from google.colab import files
    files.download(final_path)
except ImportError:
    print('Not running in Colab - find your video at', final_path)
